# Benchmark — TreeRAG vs qms-search, judged head to head

Reads `treerag_answers.json` and `qms_answers.json` (already produced by the two
systems over the full question set), then for every question scores each system's
free-text response 0–1 with the same gpt-oss judge, grading by MEANING against the
multiple-choice correct answer(s). It also lines up the three resource metrics both
files already carry: time, input tokens, output tokens.

The report has two parts:

* **summary** — mean and standard deviation of all four metrics (accuracy, time,
  input tokens, output tokens) for each system, plus paired deltas.
* **per-question** — the four metrics side by side for both systems, one row each.
  Where the two systems disagree sharply on accuracy, or a metric is anomalous, a
  short LLM-written note explains what happened.

Field mapping handled automatically:

| field        | treerag_answers.json | qms_answers.json |
|--------------|----------------------|------------------|
| id           | id                   | qid              |
| response     | treerag_response     | answer           |
| time         | time_sec             | elapsed_sec      |
| input tokens | in_tokens            | input_tokens     |
| output tokens| out_tokens           | output_tokens    |

Outputs `benchmark_report.json` and `benchmark_per_question.csv`. Only the judging
needs Ollama; the metric stats do not.

In [1]:
%pip install ollama numpy pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, re, time, math, ast
from pathlib import Path
from collections import Counter as _C
import numpy as np, pandas as pd

OLLAMA_URL  = "http://localhost:11528"
JUDGE_MODEL = "gpt-oss:120b"
TREERAG_FILE = "treerag_answers.json"
QMS_FILE     = "qms_answers.json"
REPORT_JSON  = "benchmark_report.json"
REPORT_CSV   = "benchmark_per_question.csv"
KEEP_ALIVE   = "30m"
# flag a question for an LLM explanation when the two systems' accuracy differ by at least this
ACC_GAP_FLAG = 0.5
print(f"judge {JUDGE_MODEL}; reading {TREERAG_FILE} and {QMS_FILE}")

judge gpt-oss:120b; reading treerag_answers.json and qms_answers.json


In [3]:
import ollama
client = ollama.Client(host=OLLAMA_URL, timeout=600)

def _names(r):
    raw=r.get("models",[]) if hasattr(r,"get") else getattr(r,"models",[])
    out=[]
    for m in raw:
        n=getattr(m,"model",None) or getattr(m,"name",None)
        if n is None and isinstance(m,dict): n=m.get("model") or m.get("name")
        if n: out.append(n)
    return out

def _wait():
    said=False
    while True:
        try:
            if any(JUDGE_MODEL in n for n in _names(client.list())):
                print(f"ollama ok; {JUDGE_MODEL} loaded"); return
            why=f"{JUDGE_MODEL} not loaded yet"
        except Exception as e: why=f"unreachable; {type(e).__name__}: {e}"
        if not said: print(f"waiting for ollama, {why}; rechecking every 10s"); said=True
        time.sleep(10)

# one judge/explain call; returns text, retries through tunnel drops, counts nothing
def ask(prompt, num_predict=400, think=False):
    attempt=0; pass_think=True
    while True:
        kw=dict(model=JUDGE_MODEL, messages=[{"role":"user","content":prompt}],
                options={"temperature":0,"num_predict":num_predict}, keep_alive=KEEP_ALIVE)
        if pass_think: kw["think"]=think
        try:
            r=client.chat(**kw)
            t=(r["message"]["content"] or "").strip()
            if not t:
                try: t=(r["message"]["thinking"] or "").strip()
                except Exception: pass
            return t
        except TypeError: pass_think=False
        except Exception as e:
            attempt+=1
            if attempt==1 or attempt%5==0: print(f"[waiting for ollama] {type(e).__name__}: {e}; retrying")
            time.sleep(min(60,5*2**min(attempt-1,4)))

_wait()

ollama ok; gpt-oss:120b loaded


In [4]:
# load either file tolerantly and normalise to a common record shape keyed by id
def _read(path):
    raw=Path(path).read_text(encoding="utf-8")
    try: data=json.loads(raw)
    except Exception: data=ast.literal_eval(raw)
    if isinstance(data,dict): data=data.get("questions",list(data.values()))
    return data

def _ans_list(d):
    a=d.get("correct_answers") or d.get("answers") or ([d.get("answer")] if d.get("answer") else [])
    letters={m.upper() for m in re.findall(r"(?<![A-Za-z])([A-Fa-f])(?![A-Za-z])", " ".join(map(str,a)))}
    return [L for L in "ABCDEF" if L in letters]

def _norm(d, response_key, time_key, in_key, out_key):
    qid=str(d.get("id") or d.get("qid"))
    return {"id":qid,
            "question":d.get("question") or d.get("stem") or "",
            "options":{str(k).upper():str(v) for k,v in (d.get("options") or {}).items()},
            "correct":_ans_list(d),
            "response":str(d.get(response_key) or ""),
            "time":float(d.get(time_key) or 0.0),
            "in":int(d.get(in_key) or 0),
            "out":int(d.get(out_key) or 0)}

tree_raw=_read(TREERAG_FILE); qms_raw=_read(QMS_FILE)
TREE={r["id"]:r for r in (_norm(d,"treerag_response","time_sec","in_tokens","out_tokens") for d in tree_raw)}
QMS ={r["id"]:r for r in (_norm(d,"answer","elapsed_sec","input_tokens","output_tokens") for d in qms_raw)}

ids=[i for i in TREE if i in QMS]            # judge only questions both systems answered
only_tree=[i for i in TREE if i not in QMS]; only_qms=[i for i in QMS if i not in TREE]
print(f"treerag {len(TREE)}; qms {len(QMS)}; in common {len(ids)}")
if only_tree: print(f"  only in treerag ({len(only_tree)}): {only_tree[:8]}")
if only_qms:  print(f"  only in qms ({len(only_qms)}): {only_qms[:8]}")

treerag 324; qms 324; in common 324


In [5]:
# the accuracy judge; same rubric as the eval harness, grades a free response vs the correct option(s) by meaning
def judge_accuracy(question, options, correct, response):
    multi=len(correct)>1
    opts="\n".join(f"{L}. {options[L]}" for L in "ABCDEF" if L in options)
    gold="\n".join(f"{L}. {options.get(L,'')}" for L in correct)
    prompt=("you are a fair grader. a system answered an open question in its own words and could not see the "
            "choices. the multiple choice version below has the correct option(s) marked and those are the ground "
            "truth. the correct answer may be ONE OR MORE options.\n\n"
            f"QUESTION: {question}\nOPTIONS:\n{opts}\nCORRECT OPTION(S): {', '.join(correct)}\n{gold}\n\n"
            f"SYSTEM RESPONSE:\n{response or '(empty)'}\n\n"
            "grade from 0.0 to 1.0 how well the response matches the MEANING of the correct option(s). judge by "
            "meaning not wording. "
            +("when several options are correct, give full credit only if the response conveys ALL of them, and "
              "proportional partial credit for covering some. " if multi else
              "give full or near full credit when it conveys the correct idea even in different words, partial when "
              "incomplete. ")
            +"give low credit when it matches a wrong option or is irrelevant. reply with ONLY json:\n"
            '{"score": <0.0 to 1.0>, "reason": "<one concise sentence>"}')
    return _parse(ask(prompt,num_predict=400))

def _parse(text):
    t=(text or "").strip()
    m=re.search(r'\{.*\}', t, re.S)
    if m:
        try:
            o=json.loads(m.group(0)); return max(0.0,min(1.0,float(o.get("score",0)))), str(o.get("reason",""))[:300]
        except Exception: pass
    m=re.search(r'(\d?\.\d+|\d)', t)
    return (max(0.0,min(1.0,float(m.group(1)))) if m else 0.0), "unparsed judge reply"

In [6]:
# judge both systems on every common question; resumable so a tunnel drop does not lose work
JCACHE_FILE=Path("benchmark_cache/judge.json"); JCACHE={}
if JCACHE_FILE.exists(): JCACHE=json.loads(JCACHE_FILE.read_text())
def _jsave():
    JCACHE_FILE.parent.mkdir(exist_ok=True)
    tmp=JCACHE_FILE.with_suffix(".tmp"); tmp.write_text(json.dumps(JCACHE)); tmp.replace(JCACHE_FILE)

def _bar(total, done, desc):
    try:
        from tqdm.auto import tqdm; return tqdm(total=total, initial=done, desc=desc)
    except Exception:
        class _S:
            def update(self,n=1): pass
            def close(self): pass
            def set_postfix(self,**k): pass
        return _S()

def judge_all():
    bar=_bar(len(ids)*2, sum(1 for i in ids for s in ("tree","qms") if f"{s}::{i}" in JCACHE), "judging")
    for i in ids:
        for sys,store in (("tree",TREE),("qms",QMS)):
            ck=f"{sys}::{i}"
            if ck in JCACHE: continue
            r=store[i]
            sc,why=judge_accuracy(r["question"], r["options"] or (TREE.get(i) or QMS.get(i))["options"],
                                  r["correct"] or (TREE.get(i) or QMS.get(i))["correct"], r["response"])
            JCACHE[ck]={"score":sc,"reason":why}; _jsave(); bar.update(1)
    bar.close()
    print(f"judged {len(ids)} questions x 2 systems")

judge_all()

/opt/homebrew/Cellar/jupyterlab/4.5.7_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
judging: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 648/648 [1:01:02<00:00,  5.79s/it]

judged 324 questions x 2 systems


In [7]:
# assemble the per-question table with all four metrics side by side
rows=[]
for i in ids:
    t,q=TREE[i],QMS[i]
    ts=JCACHE[f"tree::{i}"]["score"]; qs=JCACHE[f"qms::{i}"]["score"]
    rows.append({"id":i,"question":t["question"][:90],
        "tree_accuracy":ts,"qms_accuracy":qs,"accuracy_delta":round(ts-qs,3),
        "tree_time":t["time"],"qms_time":q["time"],
        "tree_in":t["in"],"qms_in":q["in"],
        "tree_out":t["out"],"qms_out":q["out"],
        "tree_reason":JCACHE[f"tree::{i}"]["reason"],"qms_reason":JCACHE[f"qms::{i}"]["reason"]})
per_q=pd.DataFrame(rows)
print(f"assembled {len(per_q)} rows")
per_q.head()

assembled 324 rows


,id,question,tree_accuracy,qms_accuracy,accuracy_delta,tree_time,qms_time,tree_in,qms_in,tree_out,qms_out,tree_reason,qms_reason
0,QMS-1,Where do you find SOPs?,0.00,0.90,-0.90,92.903,52.24,5193,467,3566,72,The response cites a file directory unrelated ...,The response correctly identifies that SOPs ar...
1,QMS-2,What version of an SOP should be used?,0.95,1.00,-0.05,253.292,20.18,7032,469,4726,76,The response correctly emphasizes using the cu...,The answer correctly identifies that the SOP v...
2,QMS-3,How do you initiate the CAPA process?,0.00,0.45,-0.45,137.536,38.85,7251,469,4798,58,unparsed judge reply,The response describes one specific way to sta...
3,QMS-4,What conditions count as a non-conformance?,0.00,0.00,0.00,176.425,16.20,8332,469,5744,54,The response defines non‑conformance generally...,The response lists various non‑conformances bu...
4,QMS-5,What is the main purpose of Proficiency Testin...,0.85,1.00,-0.15,137.824,18.76,9583,472,5646,60,The response correctly captures that PT assess...,The response accurately describes PT as evalua...


In [8]:
# summary: mean and standard deviation of all four metrics for each system, plus paired deltas
def _ms(s): return round(float(np.mean(s)),3), round(float(np.std(s,ddof=1)) if len(s)>1 else 0.0,3)
metrics={"accuracy":("tree_accuracy","qms_accuracy"),
         "time_sec":("tree_time","qms_time"),
         "input_tokens":("tree_in","qms_in"),
         "output_tokens":("tree_out","qms_out")}
summary={}
for name,(tc,qc) in metrics.items():
    tm,tsd=_ms(per_q[tc]); qm,qsd=_ms(per_q[qc])
    summary[name]={"treerag_mean":tm,"treerag_std":tsd,"qms_mean":qm,"qms_std":qsd,
                   "mean_diff_tree_minus_qms":round(tm-qm,3)}
summ_df=pd.DataFrame(summary).T[["treerag_mean","treerag_std","qms_mean","qms_std","mean_diff_tree_minus_qms"]]
print("=== summary over", len(per_q), "questions ===")
print(summ_df.to_string())
summ_df

=== summary over 324 questions ===
               treerag_mean  treerag_std  qms_mean  qms_std  mean_diff_tree_minus_qms
accuracy              0.293        0.384     0.546    0.436                    -0.253
time_sec            147.417      211.391    82.540  160.055                    64.877
input_tokens       7584.302     1690.389   474.253    6.490                  7110.049
output_tokens      5011.386     1251.992    61.062   11.852                  4950.324


,treerag_mean,treerag_std,qms_mean,qms_std,mean_diff_tree_minus_qms
accuracy,0.293,0.384,0.546,0.436,-0.253
time_sec,147.417,211.391,82.540,160.055,64.877
input_tokens,7584.302,1690.389,474.253,6.490,7110.049
output_tokens,5011.386,1251.992,61.062,11.852,4950.324


In [ ]:
# explain the notable cases: large accuracy gaps, or a metric far outside its systems own distribution
def _anoms(col):
    v=per_q[col].astype(float); mu,sd=v.mean(),v.std(ddof=1) or 1.0
    return set(per_q.loc[(v-mu).abs()>3*sd,"id"])
flag_metric=set().union(*[_anoms(c) for c in ["tree_time","qms_time","tree_in","qms_in","tree_out","qms_out"]])
flag_acc=set(per_q.loc[per_q["accuracy_delta"].abs()>=ACC_GAP_FLAG,"id"])
flagged=[i for i in ids if i in flag_acc or i in flag_metric]
print(f"{len(flagged)} questions flagged for an explanation ({len(flag_acc)} accuracy gaps, {len(flag_metric)} metric anomalies)")

# short llm note on why the two systems diverged on a flagged question
def explain(i):
    t,q=TREE[i],QMS[i]; r=next(x for x in rows if x["id"]==i)
    prompt=("two retrieval systems answered the same question and their results differ. in 1-2 sentences say why, "
            "referring to the responses and the numbers. be concrete and neutral.\n\n"
            f"QUESTION: {t['question']}\nCORRECT: {', '.join(t['correct'])}\n\n"
            f"TREERAG response: {t['response'][:600]}\nTREERAG accuracy {r['tree_accuracy']}, time {r['tree_time']}s, "
            f"in {r['tree_in']}, out {r['tree_out']}\n\n"
            f"QMS response: {q['response'][:600]}\nQMS accuracy {r['qms_accuracy']}, time {r['qms_time']}s, "
            f"in {r['qms_in']}, out {r['qms_out']}\n\nexplanation:")
    return ask(prompt,num_predict=220,think=False).strip()

notes={}
nb=_bar(len(flagged),0,"explaining")
for i in flagged: notes[i]=explain(i); nb.update(1)
nb.close()
per_q["explanation"]=per_q["id"].map(lambda i: notes.get(i,""))
for i in flagged[:12]:
    print(f"\n[{i}] tree {JCACHE[f'tree::{i}']['score']} vs qms {JCACHE[f'qms::{i}']['score']}")
    print("  "+notes[i][:300])

126 questions flagged for an explanation (117 accuracy gaps, 18 metric anomalies)


explaining:  21%|███████████████████████                                                                                         | 26/126 [02:15<08:40,  5.21s/it]

In [ ]:
# write the report; json carries summary + per-question + flagged notes, csv is the side-by-side table
report={"systems":["treerag","qms"],"n_questions":int(len(per_q)),
        "only_in_treerag":only_tree,"only_in_qms":only_qms,
        "summary":summary,
        "flagged":{i:notes[i] for i in flagged},
        "per_question":per_q.to_dict(orient="records")}
Path(REPORT_JSON).write_text(json.dumps(report,indent=2))
per_q.to_csv(REPORT_CSV,index=False)
print(f"wrote {REPORT_JSON} and {REPORT_CSV}")
print("\n=== final summary ===")
print(summ_df.to_string())